In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP & DETERMINISTIC CPU BENCHMARK HARNESS
# ============================================================
import os
import gc
import time
import random
from typing import Union, Dict, Any, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# 1. Force Pure CPU Execution & Pin Threads
DEVICE = torch.device('cpu')

# Pinning PyTorch to 1 intra-op thread eliminates OpenMP context-switching jitter
torch.set_num_threads(1)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
print(f"✓ Active Device : CPU")
print(f"✓ PyTorch Cores : {torch.get_num_threads()} Thread (Single-core deterministic baseline)")

# 2. High-Precision CPU Benchmarking Function
def benchmark_cpu(
    model: nn.Module,
    dummy_inputs: Union[torch.Tensor, Tuple[torch.Tensor, ...], Dict[str, torch.Tensor]],
    n_runs: int = 50,
    warmup: int = 15,
    trim_ratio: float = 0.10
) -> Dict[str, float]:
    """
    Measures stable single-core CPU latency using high-resolution nanosecond timers,
    disabled GC during execution, and trimmed mean filtering.
    """
    model.eval()

    def forward_pass():
        if isinstance(dummy_inputs, tuple):
            return model(*dummy_inputs)
        elif isinstance(dummy_inputs, dict):
            return model(**dummy_inputs) # Handle dictionary inputs for Hugging Face models
        return model(dummy_inputs)

    # 1. Warm-up to stabilize CPU instruction cache and branch predictors
    with torch.inference_mode():
        for _ in range(warmup):
            _ = forward_pass()

    # 2. Timed runs with GC disabled to eliminate GC collection pauses
    latencies_ns = []
    gc.disable()
    try:
        with torch.inference_mode():
            for _ in range(n_runs):
                t0 = time.perf_counter_ns()
                _ = forward_pass()
                t1 = time.perf_counter_ns()
                latencies_ns.append(t1 - t0)
    finally:
        gc.enable()

    # Convert nanoseconds to milliseconds
    latencies = np.array(latencies_ns, dtype=np.float64) / 1e6

    # 10% trimmed mean to filter out OS scheduling interrupts
    k = int(len(latencies) * trim_ratio)
    trimmed = np.sort(latencies)[k:-k] if k > 0 else latencies

    mean_ms = float(np.mean(trimmed))
    std_ms = float(np.std(trimmed))
    p50_ms = float(np.median(latencies))
    p95_ms = float(np.percentile(latencies, 95))
    fps = 1000.0 / mean_ms if mean_ms > 0 else 0.0

    return {
        "mean_ms": mean_ms,
        "std_ms": std_ms,
        "p50_ms": p50_ms,
        "p95_ms": p95_ms,
        "fps": fps
    }

✓ Active Device : CPU
✓ PyTorch Cores : 1 Thread (Single-core deterministic baseline)


In [ ]:
# ============================================================
# CELL 2: VISION BACKBONES CPU LATENCY & PARAMETER BENCHMARK
# ============================================================
import torchvision.models as models
from torchvision.models import (
    resnet50, ResNet50_Weights,
    densenet121, DenseNet121_Weights,
    vit_b_16, ViT_B_16_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)

# Define Vision Model Constructors
vision_models = {
    "ResNet-50": lambda: resnet50(weights=ResNet50_Weights.IMAGENET1K_V2),
    "DenseNet-121": lambda: densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1),
    "ViT-B/16": lambda: vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1),
    "EfficientNet-B0": lambda: efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
}

# Static Pre-allocated Input (Batch size 1, 3 channels, 224x224)
dummy_img = torch.randn(1, 3, 224, 224, device=DEVICE)

vision_benchmark_results = []

print("\n" + "="*95)
print(f"{'Backbone':<16} | {'Params (M)':<10} | {'Mean Latency (ms)':<18} | {'Median p50 (ms)':<16} | {'Throughput (FPS)':<16}")
print("="*95)

for name, model_constructor in vision_models.items():
    # Instantiate model on CPU
    model = model_constructor().to(DEVICE).eval()

    # Calculate parameter count (in millions)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    # Run deterministic CPU benchmark
    stats = benchmark_cpu(model, dummy_img, n_runs=50, warmup=15)

    vision_benchmark_results.append({
        "Model": name,
        "Parameters (M)": f"{params_m:.2f}M",
        "Mean Latency (ms)": f"{stats['mean_ms']:.2f} ± {stats['std_ms']:.2f}",
        "Median p50 (ms)": f"{stats['p50_ms']:.2f}",
        "p95 Latency (ms)": f"{stats['p95_ms']:.2f}",
        "Throughput (FPS)": f"{stats['fps']:.2f}"
    })

    print(f"{name:<16} | {params_m:6.2f}M    | {stats['mean_ms']:6.2f} ± {stats['std_ms']:5.2f} ms | {stats['p50_ms']:8.2f} ms     | {stats['fps']:8.2f} FPS")

    # Clean memory between runs
    del model
    gc.collect()

print("="*95)

# Summary DataFrame for Publication Table Export
df_vision = pd.DataFrame(vision_benchmark_results)
print("\nFinal Vision Summary Table:")
print(df_vision.to_string(index=False))



Backbone         | Params (M) | Mean Latency (ms)  | Median p50 (ms)  | Throughput (FPS)
ResNet-50        |  25.56M    | 208.54 ± 83.69 ms |   178.45 ms     |     4.80 FPS
DenseNet-121     |   7.98M    | 128.34 ± 30.84 ms |   112.22 ms     |     7.79 FPS
ViT-B/16         |  86.57M    | 502.91 ± 129.45 ms |   493.08 ms     |     1.99 FPS
EfficientNet-B0  |   5.29M    | 106.59 ± 33.42 ms |   109.97 ms     |     9.38 FPS

Final Vision Summary Table:
          Model Parameters (M) Mean Latency (ms) Median p50 (ms) p95 Latency (ms) Throughput (FPS)
      ResNet-50         25.56M    208.54 ± 83.69          178.45           811.14             4.80
   DenseNet-121          7.98M    128.34 ± 30.84          112.22           209.40             7.79
       ViT-B/16         86.57M   502.91 ± 129.45          493.08          1000.93             1.99
EfficientNet-B0          5.29M    106.59 ± 33.42          109.97           171.70             9.38


In [ ]:
# ============================================================
# CELL 3: TEXT BACKBONES CPU BENCHMARK
# ============================================================
from transformers import AutoTokenizer, AutoModel

text_models = {
    "BERT-base": "bert-base-uncased",
    "RoBERTa-base": "roberta-base",
    "BERTweet-base": "vinai/bertweet-base",
    "TinyBERT": "huawei-noah/TinyBERT_General_4L_312D"
}

sample_text = "IRAN'S EARTHQUAKE EXPOSES POLITICAL RIFTS AND INEFFECTIVE GOVERNANCE"
MAX_TEXT_LENGTH = 64
text_benchmark_results = []
text_results_dict = {}

print("\n" + "="*95)
print(f"{'Backbone':<16} | {'Params (M)':<10} | {'Mean Latency (ms)':<18} | {'Median p50 (ms)':<16} | {'Throughput (FPS)':<16}")
print("="*95)

for name, hf_name in text_models.items():
    tok = AutoTokenizer.from_pretrained(hf_name)
    mdl = AutoModel.from_pretrained(hf_name).to(DEVICE).eval()

    # Pre-tokenize input outside the timing window to isolate pure forward-pass latency
    enc = tok(sample_text, return_tensors='pt', padding='max_length', truncation=True, max_length=MAX_TEXT_LENGTH)
    static_txt_inputs = {k: v.to(DEVICE) for k, v in enc.items()}

    params_m = sum(p.numel() for p in mdl.parameters()) / 1e6
    stats = benchmark_cpu(mdl, static_txt_inputs, n_runs=50, warmup=10)

    text_results_dict[name] = stats
    text_benchmark_results.append({
        "Model": name,
        "Parameters (M)": f"{params_m:.2f}M",
        "Mean Latency (ms)": f"{stats['mean_ms']:.2f} ± {stats['std_ms']:.2f}",
        "Median p50 (ms)": f"{stats['p50_ms']:.2f}",
        "p95 Latency (ms)": f"{stats['p95_ms']:.2f}",
        "Throughput (FPS)": f"{stats['fps']:.2f}"
    })

    print(f"{name:<16} | {params_m:6.2f}M    | {stats['mean_ms']:6.2f} ± {stats['std_ms']:5.2f} ms | {stats['p50_ms']:8.2f} ms     | {stats['fps']:8.2f} FPS")

    del mdl, tok
    gc.collect()

print("="*95)
df_text = pd.DataFrame(text_benchmark_results)
print("\nFinal Text Summary Table:")
print(df_text.to_string(index=False))


Backbone         | Params (M) | Mean Latency (ms)  | Median p50 (ms)  | Throughput (FPS)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT-base        | 109.48M    | 268.70 ± 69.10 ms |   243.89 ms     |     3.72 FPS


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTweet-base    | 134.90M    | 172.39 ± 24.90 ms |   156.55 ms     |     5.80 FPS


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TinyBERT         |  14.35M    |   9.95 ±  0.68 ms |     9.72 ms     |   100.48 FPS

Final Text Summary Table:
        Model Parameters (M) Mean Latency (ms) Median p50 (ms) p95 Latency (ms) Throughput (FPS)
    BERT-base        109.48M    268.70 ± 69.10          243.89           896.41             3.72
 RoBERTa-base        124.65M    184.45 ± 27.19          180.52           232.87             5.42
BERTweet-base        134.90M    172.39 ± 24.90          156.55           217.30             5.80
     TinyBERT         14.35M       9.95 ± 0.68            9.72            12.96           100.48


In [ ]:
# ============================================================
# CELL 4: BELT ARCHITECTURE & COMPONENT DEFINITIONS
# ============================================================
IMAGE_DIM = 1280
TEXT_DIM = 312
HIDDEN_DIM = 256
EMBED_DIM = 64
NUM_HEADS = 4
DROPOUT = 0.1
MAX_TEXT_LENGTH = 64

class CoralHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 3):
        super().__init__()
        self.num_thresholds = max(num_classes - 1, 1)
        self.fc = nn.Linear(in_dim, 1, bias=False)
        self.bias0 = nn.Parameter(torch.zeros(1))
        self.deltas = nn.Parameter(torch.ones(max(self.num_thresholds - 1, 0)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.fc(x)
        biases = [self.bias0]
        b = self.bias0
        for i in range(self.num_thresholds - 1):
            b = b - F.softplus(self.deltas[i])
            biases.append(b)
        biases = torch.cat(biases)
        return base + biases.unsqueeze(0)


class AdapterPoolText(nn.Module):
    def __init__(self, input_dim: int = TEXT_DIM, output_dim: int = HIDDEN_DIM, bottleneck: int = 64):
        super().__init__()
        self.down = nn.Linear(input_dim, bottleneck)
        self.up = nn.Linear(bottleneck, input_dim)
        self.norm = nn.LayerNorm(input_dim)
        self.projection = nn.Linear(input_dim, output_dim)

    def forward(self, text: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        residual = text
        x = F.gelu(self.down(text))
        x = self.up(x)
        x = self.norm(residual + x)
        mask_expanded = mask.unsqueeze(-1)
        x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-6)
        return self.projection(x)


class FlexibleEnsembleMember(nn.Module):
    """Ablation architecture supporting Concat, Unidirectional, Bidirectional, and BELT fusion."""
    def __init__(self, fusion_type: str = "belt", num_classes: int = 2):
        super().__init__()
        assert fusion_type in ("concat", "uni", "bi", "belt")
        self.fusion_type = fusion_type

        self.image_projection = nn.Sequential(
            nn.Linear(IMAGE_DIM, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.ReLU()
        )
        self.text_projection = AdapterPoolText()

        if fusion_type in ("uni", "bi", "belt"):
            self.image_to_text = nn.MultiheadAttention(HIDDEN_DIM, NUM_HEADS, dropout=DROPOUT, batch_first=True)
            self.image_norm = nn.LayerNorm(HIDDEN_DIM)

        if fusion_type in ("bi", "belt"):
            self.text_to_image = nn.MultiheadAttention(HIDDEN_DIM, NUM_HEADS, dropout=DROPOUT, batch_first=True)
            self.text_norm = nn.LayerNorm(HIDDEN_DIM)

        if fusion_type == "belt":
            self.self_attention = nn.TransformerEncoderLayer(
                d_model=HIDDEN_DIM, nhead=NUM_HEADS, dim_feedforward=512,
                dropout=DROPOUT, batch_first=True, norm_first=True
            )

        self.embedding = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, EMBED_DIM),
            nn.LayerNorm(EMBED_DIM),
            nn.ReLU()
        )
        self.coral_head = CoralHead(EMBED_DIM, num_classes=num_classes)

    def embed(self, image: torch.Tensor, text: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        if image.ndim == 4:
            image = F.adaptive_avg_pool2d(image, 1).flatten(1)
        v = self.image_projection(image)
        t = self.text_projection(text, mask)

        if self.fusion_type == "concat":
            return self.embedding(torch.cat([v, t], dim=1))

        v_tok, t_tok = v.unsqueeze(1), t.unsqueeze(1)
        v_att, _ = self.image_to_text(v_tok, t_tok, t_tok)

        if self.fusion_type == "uni":
            v_new = self.image_norm(v_tok + v_att)
            fused = torch.cat([v_new.squeeze(1), t_tok.squeeze(1)], dim=1)
            return self.embedding(fused)

        t_att, _ = self.text_to_image(t_tok, v_tok, v_tok)
        v_new = self.image_norm(v_tok + v_att)
        t_new = self.text_norm(t_tok + t_att)

        if self.fusion_type == "bi":
            fused = torch.cat([v_new.squeeze(1), t_new.squeeze(1)], dim=1)
            return self.embedding(fused)

        # BELT: Bidirectional Cross-Attention + Self-Attention Refinement
        pair = torch.cat([v_new, t_new], dim=1)
        pair = self.self_attention(pair)
        fused = pair.flatten(start_dim=1)
        return self.embedding(fused)

    def forward(self, image: torch.Tensor, text: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.coral_head(self.embed(image, text, mask))

print("✓ Architecture modules compiled successfully.")

✓ Architecture modules compiled successfully.


In [ ]:
import torch.nn.functional as F
# ============================================================
# CELL 5: FUSION PARADIGM LATENCY ABLATION (CPU)
# ============================================================

# Wrapper module to benchmark the .embed method of FlexibleEnsembleMember
class EmbedWrapper(nn.Module):
    def __init__(self, ensemble_member_instance):
        super().__init__()
        self.member = ensemble_member_instance

    def forward(self, image, text, mask):
        return self.member.embed(image, text, mask)

fusion_configs = {
    "Concat": "concat",
    "Unidirectional": "uni",
    "Bidirectional": "bi",
    "BELT (Ours)": "belt"
}

# Pre-allocated intermediate feature tensors matching backbone outputs
dummy_v = torch.randn(1, IMAGE_DIM, device=DEVICE)
dummy_t = torch.randn(1, MAX_TEXT_LENGTH, TEXT_DIM, device=DEVICE)
dummy_m = torch.ones(1, MAX_TEXT_LENGTH, device=DEVICE)
static_fusion_inputs = (dummy_v, dummy_t, dummy_m)

fusion_benchmark_results = []

print("\n" + "="*105)
print(f"{'Fusion Paradigm':<18} | {'Params (M)':<10} | {'Mean Latency (ms)':<18} | {'Median p50 (ms)':<16} | {'Throughput (FPS)':<16}")
print("="*105)

for label, ftype in fusion_configs.items():
    member = FlexibleEnsembleMember(fusion_type=ftype).to(DEVICE).eval()
    embed_model = EmbedWrapper(member).to(DEVICE).eval() # Wrap the member for benchmarking

    # Calculate parameter count for the member (excluding the CoralHead as embed() is benchmarked)
    # The member here includes image_projection, text_projection, attention layers, and embedding
    params_m = sum(p.numel() for p in member.parameters()) / 1e6

    stats = benchmark_cpu(embed_model, static_fusion_inputs, n_runs=100, warmup=20)

    fusion_benchmark_results.append({
        "Fusion Strategy": label,
        "Parameters (M)": f"{params_m:.2f}M",
        "Mean Latency (ms)": f"{stats['mean_ms']:.3f} \u00b1 {stats['std_ms']:.3f}",
        "Median p50 (ms)": f"{stats['p50_ms']:.3f}",
        "p95 Latency (ms)": f"{stats['p95_ms']:.3f}",
        "Throughput (FPS)": f"{stats['fps']:.1f}"
    })

    print(f"{label:<18} | {params_m:6.2f}M    | {stats['mean_ms']:6.3f} \u00b1 {stats['std_ms']:5.3f} ms | {stats['p50_ms']:8.3f} ms     | {stats['fps']:8.1f} FPS")

    del embed_model # Clean up wrapper
    del member
    gc.collect()

print("="*105)
df_fusion = pd.DataFrame(fusion_benchmark_results)
print(df_fusion.to_string(index=False))


Fusion Paradigm    | Params (M) | Mean Latency (ms)  | Median p50 (ms)  | Throughput (FPS)
Concat             |   0.48M    |  0.531 ± 0.039 ms |    0.524 ms     |   1883.1 FPS
Unidirectional     |   0.75M    |  0.887 ± 0.047 ms |    0.880 ms     |   1127.7 FPS
Bidirectional      |   1.01M    |  1.220 ± 0.066 ms |    1.194 ms     |    819.5 FPS
BELT (Ours)        |   1.54M    |  1.575 ± 0.058 ms |    1.564 ms     |    634.8 FPS
Fusion Strategy Parameters (M) Mean Latency (ms) Median p50 (ms) p95 Latency (ms) Throughput (FPS)
         Concat          0.48M     0.531 ± 0.039           0.524            0.633           1883.1
 Unidirectional          0.75M     0.887 ± 0.047           0.880            1.043           1127.7
  Bidirectional          1.01M     1.220 ± 0.066           1.194            1.447            819.5
    BELT (Ours)          1.54M     1.575 ± 0.058           1.564            1.752            634.8


In [ ]:
# ============================================================
# CELL 6: CLASSIFIER HEAD LATENCY (BCE vs CORAL ON CPU)
# ============================================================
dummy_z = torch.randn(1, EMBED_DIM, device=DEVICE)
num_classes_list = [2, 3]

head_benchmark_results = []

print("\n" + "="*80)
print(f"{'Head Type':<16} | {'Params':<8} | {'Mean Latency (μs)':<20} | {'Median p50 (μs)':<16}")
print("="*80)

for classes in num_classes_list:
    heads = [
        (f"BCE (K={classes})", nn.Linear(EMBED_DIM, 1)),
        (f"CORAL (K={classes})", CoralHead(EMBED_DIM, num_classes=classes))
    ]
    for name, head in heads:
        head = head.to(DEVICE).eval()
        params = sum(p.numel() for p in head.parameters())
        stats = benchmark_cpu(head, dummy_z, n_runs=200, warmup=30)

        # Converted to microseconds (μs) for clearer tabular presentation
        head_benchmark_results.append({
            "Head": name,
            "Parameters": params,
            "Mean Latency (μs)": f"{stats['mean_ms']*1000:.2f} ± {stats['std_ms']*1000:.2f}",
            "Median (μs)": f"{stats['p50_ms']*1000:.2f}"
        })

        print(f"{name:<16} | {params:<8} | {stats['mean_ms']*1000:7.2f} ± {stats['std_ms']*1000:6.2f} μs | {stats['p50_ms']*1000:8.2f} μs")

        del head
        gc.collect()

print("="*80)


Head Type        | Params   | Mean Latency (μs)    | Median p50 (μs) 
BCE (K=2)        | 65       |   13.29 ±   0.62 μs |    13.15 μs
CORAL (K=2)      | 65       |   27.70 ±   1.67 μs |    27.26 μs
BCE (K=3)        | 65       |   15.18 ±   0.65 μs |    15.26 μs
CORAL (K=3)      | 66       |   35.02 ±   9.65 μs |    30.45 μs


In [ ]:
from google.colab import drive

# ------------------------------------------------------------
# 1. Mount Drive and Setup Paths & CPU Configuration
# ------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/SeaBERT_Final'
# Choose whichever checkpoint you want to benchmark (e.g. inform or severity)
CHECKPOINT_PATH = f'{BASE}/checkpoints/belt_severity_aug.pth'
# Alternatively: CHECKPOINT_PATH = f'{BASE}/checkpoints/belt_severity_aug.pth'

DEVICE = torch.device('cpu')
torch.set_num_threads(1)  # Single-core deterministic baseline for CPU benchmarking

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

# Hyperparameters matching your BELT config
IMAGE_DIM = 1280
TEXT_DIM = 312
HIDDEN_DIM = 256
EMBED_DIM = 64
NUM_HEADS = 4
DROPOUT = 0.1
MAX_TEXT_LENGTH = 64
NUM_MEMBERS = 5

# ------------------------------------------------------------
# 2. Decoupled BELT Architecture (Loss/Head Independent)
# ------------------------------------------------------------
class AdapterPoolText(nn.Module):
    def __init__(self, input_dim=TEXT_DIM, output_dim=HIDDEN_DIM, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(input_dim, bottleneck)
        self.up = nn.Linear(bottleneck, input_dim)
        self.norm = nn.LayerNorm(input_dim)
        self.projection = nn.Linear(input_dim, output_dim)

    def forward(self, text: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        residual = text
        x = F.gelu(self.down(text))
        x = self.up(x)
        x = self.norm(residual + x)
        mask = mask.unsqueeze(-1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        return self.projection(x)


class BELTBackbone(nn.Module):
    """Single ensemble member without hardcoded loss/decision head."""
    def __init__(self):
        super().__init__()
        self.image_projection = nn.Sequential(
            nn.Linear(IMAGE_DIM, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.ReLU()
        )
        self.text_projection = AdapterPoolText()
        self.image_to_text = nn.MultiheadAttention(HIDDEN_DIM, NUM_HEADS, dropout=DROPOUT, batch_first=True)
        self.text_to_image = nn.MultiheadAttention(HIDDEN_DIM, NUM_HEADS, dropout=DROPOUT, batch_first=True)
        self.image_norm = nn.LayerNorm(HIDDEN_DIM)
        self.text_norm = nn.LayerNorm(HIDDEN_DIM)
        self.self_attention = nn.TransformerEncoderLayer(
            d_model=HIDDEN_DIM, nhead=NUM_HEADS, dim_feedforward=512,
            dropout=DROPOUT, batch_first=True, norm_first=True
        )
        self.embedding = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, EMBED_DIM),
            nn.LayerNorm(EMBED_DIM),
            nn.ReLU()
        )

    def forward(self, image: torch.Tensor, text: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        if image.ndim == 4:
            image = F.adaptive_avg_pool2d(image, 1).flatten(1)
        v = self.image_projection(image)
        t = self.text_projection(text, mask)

        v_tok, t_tok = v.unsqueeze(1), t.unsqueeze(1)
        v_att, _ = self.image_to_text(v_tok, t_tok, t_tok)
        t_att, _ = self.text_to_image(t_tok, v_tok, v_tok)

        v_new = self.image_norm(v_tok + v_att)
        t_new = self.text_norm(t_tok + t_att)

        pair = torch.cat([v_new, t_new], dim=1)
        pair = self.self_attention(pair)
        return self.embedding(pair.flatten(start_dim=1))


class HeadlessEmbeddingEnsemble(nn.Module):
    """5-Member ensemble returning the weighted fused 64-D multimodal embedding."""
    def __init__(self, num_members: int = NUM_MEMBERS):
        super().__init__()
        self.members = nn.ModuleList([BELTBackbone() for _ in range(num_members)])
        self.weight_logits = nn.Parameter(torch.zeros(num_members))

    def get_weights(self) -> torch.Tensor:
        return torch.softmax(self.weight_logits, dim=0)

    def forward(self, image: torch.Tensor, text: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        embeddings = torch.stack([m(image, text, mask) for m in self.members], dim=0)
        weights = self.get_weights().view(-1, 1, 1)
        return (embeddings * weights).sum(dim=0)


# ------------------------------------------------------------
# 3. High-Precision CPU Benchmarking Function
# ------------------------------------------------------------
def benchmark_cpu(
    model: nn.Module,
    dummy_inputs: Tuple[torch.Tensor, ...],
    n_runs: int = 100,
    warmup: int = 25,
    trim_ratio: float = 0.10
) -> Dict[str, float]:
    model.eval()

    # Warmup
    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(*dummy_inputs)

    latencies_ns = []
    gc.disable()
    try:
        with torch.inference_mode():
            for _ in range(n_runs):
                t0 = time.perf_counter_ns()
                _ = model(*dummy_inputs)
                t1 = time.perf_counter_ns()
                latencies_ns.append(t1 - t0)
    finally:
        gc.enable()

    latencies = np.array(latencies_ns, dtype=np.float64) / 1e6  # to ms
    k = int(len(latencies) * trim_ratio)
    trimmed = np.sort(latencies)[k:-k] if k > 0 else latencies

    mean_ms = float(np.mean(trimmed))
    std_ms = float(np.std(trimmed))
    p50_ms = float(np.median(latencies))
    p95_ms = float(np.percentile(latencies, 95))
    fps = 1000.0 / mean_ms if mean_ms > 0 else 0.0

    return {
        "mean_ms": mean_ms,
        "std_ms": std_ms,
        "p50_ms": p50_ms,
        "p95_ms": p95_ms,
        "fps": fps
    }


# ------------------------------------------------------------
# 4. Load Weights from Google Drive & Execute Benchmark
# ------------------------------------------------------------
try:
    ensemble_model = HeadlessEmbeddingEnsemble(num_members=NUM_MEMBERS).to(DEVICE).eval()

    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint file not found at: {CHECKPOINT_PATH}")

    # Load weights with strict=False to bypass task-specific heads (coral/classifier)
    state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    missing, unexpected = ensemble_model.load_state_dict(state_dict, strict=False)

    print("\n" + "="*70)
    print("WEIGHT IMPORT STATUS")
    print("="*70)
    print(f"✓ Loaded Checkpoint : {os.path.basename(CHECKPOINT_PATH)}")
    print(f"✓ Weights Loaded    : Successfully loaded all {NUM_MEMBERS} BELT members.")
    print(f"✓ Ignored Heads     : {[k for k in unexpected if 'coral' in k or 'head' in k]}")

    # Static pre-allocated dummy inputs matching the backbone dimensions
    static_inputs = (
        torch.randn(1, IMAGE_DIM, device=DEVICE),
        torch.randn(1, MAX_TEXT_LENGTH, TEXT_DIM, device=DEVICE),
        torch.ones(1, MAX_TEXT_LENGTH, device=DEVICE)
    )

    # Run Benchmark on CPU
    stats = benchmark_cpu(ensemble_model, static_inputs, n_runs=100, warmup=25)
    params_m = sum(p.numel() for p in ensemble_model.parameters()) / 1e6

    print("\n" + "="*70)
    print("5-MEMBER BELT ENSEMBLE CPU BENCHMARK RESULTS")
    print("="*70)
    print(f"Architecture Parameters : {params_m:.2f}M")
    print(f"Mean Latency            : {stats['mean_ms']:.2f} ± {stats['std_ms']:.2f} ms")
    print(f"Median Latency (p50)    : {stats['p50_ms']:.2f} ms")
    print(f"95th Percentile (p95)   : {stats['p95_ms']:.2f} ms")
    print(f"CPU Throughput          : {stats['fps']:.2f} inferences/sec (FPS)")
    print("="*70)

    # Verify forward output
    with torch.inference_mode():
        fused_embedding = ensemble_model(*static_inputs)
        print(f"✓ Output Representation Shape: {tuple(fused_embedding.shape)} (Expected: (1, {EMBED_DIM}))")

except Exception as e:
    print(f"Error during execution: {e}")



WEIGHT IMPORT STATUS
✓ Loaded Checkpoint : belt_severity_aug.pth
✓ Weights Loaded    : Successfully loaded all 5 BELT members.
✓ Ignored Heads     : ['head.bias0', 'head.deltas', 'head.fc.weight', 'members.0.coral_head.bias0', 'members.0.coral_head.deltas', 'members.0.coral_head.fc.weight', 'members.1.coral_head.bias0', 'members.1.coral_head.deltas', 'members.1.coral_head.fc.weight', 'members.2.coral_head.bias0', 'members.2.coral_head.deltas', 'members.2.coral_head.fc.weight', 'members.3.coral_head.bias0', 'members.3.coral_head.deltas', 'members.3.coral_head.fc.weight', 'members.4.coral_head.bias0', 'members.4.coral_head.deltas', 'members.4.coral_head.fc.weight']

5-MEMBER BELT ENSEMBLE CPU BENCHMARK RESULTS
Architecture Parameters : 7.68M
Mean Latency            : 12.62 ± 1.14 ms
Median Latency (p50)    : 12.19 ms
95th Percentile (p95)   : 19.96 ms
CPU Throughput          : 79.25 inferences/sec (FPS)
✓ Output Representation Shape: (1, 64) (Expected: (1, 64))


In [ ]:
# ============================================================
# EXPORT ALL BENCHMARK RESULTS TO CSV
# ============================================================
import os
import pandas as pd

# Define output directory in your Drive
OUTPUT_DIR = f'{BASE}/results' if 'BASE' in globals() else '/content/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Vision Backbones DataFrame
# ------------------------------------------------------------
df_vision = pd.DataFrame([
    {
        "Component": "Vision Backbone",
        "Model": name,
        "Parameters (M)": round(vision_results[name]["params_M"], 2) if "params_M" in vision_results[name] else None,
        "Mean Latency (ms)": round(vision_results[name]["mean_ms"], 2),
        "Std Latency (ms)": round(vision_results[name]["std_ms"], 2),
        "Median p50 (ms)": round(vision_results[name]["p50_ms"], 2),
        "p95 Latency (ms)": round(vision_results[name]["p95_ms"], 2),
        "Throughput (FPS)": round(vision_results[name]["fps"], 2) if "fps" in vision_results[name] else round(vision_results[name]["throughput_fps"], 2)
    }
    for name in vision_results
]) if 'vision_results' in globals() and vision_results else pd.DataFrame()

# ------------------------------------------------------------
# 2. Text Backbones DataFrame
# ------------------------------------------------------------
df_text = pd.DataFrame(text_benchmark_results) if 'text_benchmark_results' in globals() and text_benchmark_results else pd.DataFrame()

# ------------------------------------------------------------
# 3. Fusion Ablation DataFrame
# ------------------------------------------------------------
df_fusion = pd.DataFrame(fusion_benchmark_results) if 'fusion_benchmark_results' in globals() and fusion_benchmark_results else pd.DataFrame()

# ------------------------------------------------------------
# 4. Classifier Heads DataFrame
# ------------------------------------------------------------
df_heads = pd.DataFrame(head_benchmark_results) if 'head_benchmark_results' in globals() and head_benchmark_results else pd.DataFrame()

# ------------------------------------------------------------
# 5. 5-Member BELT Ensemble DataFrame
# ------------------------------------------------------------
df_ensemble = pd.DataFrame([
    {
        "Component": "Ensemble Fusion",
        "Model": "5-Member BELT Backbone",
        "Parameters (M)": round(params_m, 2) if 'params_m' in globals() else None,
        "Mean Latency (ms)": round(stats["mean_ms"], 3) if 'stats' in globals() else (round(ens_stats["mean_ms"], 3) if 'ens_stats' in globals() else None),
        "Std Latency (ms)": round(stats["std_ms"], 3) if 'stats' in globals() else (round(ens_stats["std_ms"], 3) if 'ens_stats' in globals() else None),
        "Median p50 (ms)": round(stats["p50_ms"], 3) if 'stats' in globals() else (round(ens_stats["p50_ms"], 3) if 'ens_stats' in globals() else None),
        "p95 Latency (ms)": round(stats["p95_ms"], 3) if 'stats' in globals() else (round(ens_stats["p95_ms"], 3) if 'ens_stats' in globals() else None),
        "Throughput (FPS)": round(stats["fps"], 1) if 'stats' in globals() and "fps" in stats else (round(ens_stats["fps"], 1) if 'ens_stats' in globals() else None)
    }
]) if ('stats' in globals() or 'ens_stats' in globals()) else pd.DataFrame()

# ------------------------------------------------------------
# 6. End-to-End Pipeline Summary DataFrame
# ------------------------------------------------------------
df_e2e = pd.DataFrame(summary_data) if 'summary_data' in globals() and summary_data else (pd.DataFrame(summary) if 'summary' in globals() and summary else pd.DataFrame())

# ------------------------------------------------------------
# Save to Individual CSV Files
# ------------------------------------------------------------
csv_files = {
    "vision_backbones_benchmark.csv": df_vision,
    "text_backbones_benchmark.csv": df_text,
    "fusion_ablation_benchmark.csv": df_fusion,
    "classifier_heads_benchmark.csv": df_heads,
    "belt_ensemble_benchmark.csv": df_ensemble,
    "end_to_end_pipeline_summary.csv": df_e2e
}

print("="*75)
print(f"EXPORTING BENCHMARK RESULTS TO: {OUTPUT_DIR}")
print("="*75)

for filename, df in csv_files.items():
    if not df.empty:
        file_path = os.path.join(OUTPUT_DIR, filename)
        df.to_csv(file_path, index=False)
        print(f"✓ Saved: {filename} ({len(df)} rows)")
    else:
        print(f"⚠ Skipped: {filename} (DataFrame empty or variables not found)")

# ------------------------------------------------------------
# Save Master Consolidated CSV (All Components in One Table)
# ------------------------------------------------------------
valid_dfs = [df for df in [df_vision, df_text, df_fusion, df_ensemble] if not df.empty]
if valid_dfs:
    df_master = pd.concat(valid_dfs, ignore_index=True)
    master_path = os.path.join(OUTPUT_DIR, "all_benchmarks_master.csv")
    df_master.to_csv(master_path, index=False)
    print(f"\n✓ Master Consolidated CSV Saved: all_benchmarks_master.csv ({len(df_master)} total rows)")

print("="*75)

EXPORTING BENCHMARK RESULTS TO: /content/drive/MyDrive/SeaBERT_Final/results
✓ Saved: vision_backbones_benchmark.csv (4 rows)
✓ Saved: text_backbones_benchmark.csv (4 rows)
✓ Saved: fusion_ablation_benchmark.csv (4 rows)
✓ Saved: classifier_heads_benchmark.csv (4 rows)
✓ Saved: belt_ensemble_benchmark.csv (1 rows)
⚠ Skipped: end_to_end_pipeline_summary.csv (DataFrame empty or variables not found)

✓ Master Consolidated CSV Saved: all_benchmarks_master.csv (13 total rows)
